# Namma Space — Autonomous 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Namma Space Challenge (Round 1)**

This notebook trains an authentic 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### Pre-Flight Checklist:
1. **Runtime Type**: In the top menu, go to **Runtime → Change runtime type** → select **T4 GPU** → click **Save**.
2. **Uploaded Video**: A 45–75 second MP4/MOV recorded following `docs/CAPTURE_SOP.md` (1080p landscape, locked AE/AF, dual-loop).

### Step 1: Initialize Python 3.10 Environment (Bypasses Python 3.13 CUDA / C++ Compilation Errors)
Modern Colab defaults to Python 3.13, which breaks legacy CUDA extensions. This cell automatically sets up Python 3.10.

In [ ]:
# Install CondaColab to switch to Python 3.10
!pip install -q condacolab
import condacolab
condacolab.install()

# NOTE: Colab will display 'Your session crashed and has recovered'.
# This is completely normal and means the Python 3.10 environment has initialized!

### Step 2: Install PyTorch, Pre-Compiled CUDA Rasterizers & Roomsplat (Instant — No C++ Compilation!)
This cell installs official pre-compiled binary wheels for `diff-gaussian-rasterization` and `simple-knn` directly from HuggingFace, completely eliminating nvcc compiler errors.

In [ ]:
import os, sys

# 1. Install PyTorch with CUDA in Python 3.10
!conda install -y pytorch torchvision pytorch-cuda=11.8 -c pytorch -c nvidia

# 2. Install pre-compiled binary wheels for diff-gaussian-rasterization and simple-knn (instant!)
!pip install https://huggingface.co/spaces/dylanebert/LGM-mini/resolve/main/wheel/diff_gaussian_rasterization-0.0.0-cp310-cp310-linux_x86_64.whl
!pip install https://huggingface.co/spaces/dylanebert/LGM-mini/resolve/main/wheel/simple_knn-0.0.0-cp310-cp310-linux_x86_64.whl

# 3. Clone and install Roomsplat
%cd /content
!git clone https://github.com/Ahmedn1/roomsplat.git
%cd /content/roomsplat
!pip install -e . pycolmap typer rich opencv-python-headless plyfile tqdm scipy

# 4. Clone official Inria gaussian-splatting and link config
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!python -c "from roomsplat.cli import app; app()" config --gs-dir /content/gaussian-splatting

print("\nSUCCESS: Python 3.10 environment ready with pre-compiled CUDA rasterizers!")

### Step 3: Run Direct 3DGS Training on Your Extracted Room Poses
Frames and COLMAP camera poses for your video are already calculated in `/content/test_vid1_splat/`. This runs 3D Gaussian Splatting directly.

In [ ]:
%cd /content/gaussian-splatting

# Auto-detect output splat directory with existing poses
import glob, os
splat_dirs = [d for d in glob.glob("/content/*_splat") if os.path.exists(os.path.join(d, "sparse"))]
if splat_dirs:
    source_dir = splat_dirs[0]
else:
    source_dir = "/content/test_vid1_splat"

print(f"Training 3D Gaussian Splatting from: {source_dir}")
print("Running 30,000 iterations on NVIDIA T4 GPU (approx 20 min)...\n")

!python train.py \
    -s "{source_dir}" \
    -m "{source_dir}/model" \
    --iterations 30000 \
    --disable_viewer \
    --densify_until_iter 8000 \
    --densification_interval 200

print("\nSUCCESS: 3DGS training complete!")

### Step 4: Export `scene.splat` & `cameras.json` and Download
Converts your trained model to the 32-byte web binary format and downloads `namma_space_assets.zip`.

In [ ]:
%cd /content/roomsplat
import os, sys, shutil, glob
from pathlib import Path

sys.path.insert(0, "/content/roomsplat")
from roomsplat.pipeline.export import ply_to_splat, write_cameras_json
from google.colab import files

splat_dirs = [d for d in glob.glob("/content/*_splat") if os.path.exists(os.path.join(d, "sparse"))]
out_dir = Path(splat_dirs[0]) if splat_dirs else Path("/content/test_vid1_splat")
sparse_dir = out_dir / "sparse" / "0"
viewer_dir = out_dir / "viewer"
viewer_dir.mkdir(parents=True, exist_ok=True)

# Find trained PLY
ply_files = list(out_dir.glob("model/point_cloud/iteration_*/point_cloud.ply"))
if not ply_files:
    raise FileNotFoundError(f"point_cloud.ply not found in {out_dir}/model! Did training finish?")

ply_path = sorted(ply_files, key=lambda p: int(p.parent.name.split('_')[1]))[-1]
print(f"Found trained point cloud: {ply_path}")

splat_path = viewer_dir / "scene.splat"
ply_to_splat(ply_path, splat_path)
write_cameras_json(sparse_dir, viewer_dir)

# Package for download
shutil.copy(splat_path, "/content/scene.splat")
shutil.copy(viewer_dir / "cameras.json", "/content/cameras.json")
!zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json

print("\nSUCCESS: Downloading namma_space_assets.zip to your computer...")
files.download("/content/namma_space_assets.zip")